# Corponi Replication: SSL Phase 1 (Multisource Pre-training)

## Overview
This notebook implements the foundational Self-Supervised Learning (SSL) phase of the Corponi architecture. 
We use the entire collection of unlabeled multisource data for pre-training. 

### Goal
Train a robust encoder that captures physiological signal patterns across diverse contexts before fine-tuning on the labeled WESAD dataset.

In [ ]:
# Environment Setup
import os, sys, shutil, glob, pickle, zipfile
from IPython.display import FileLink

WORK_DIR = '/kaggle/working/e4selflearning'
if not os.path.exists(WORK_DIR):
    print("Cloning repository...")
    !git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

# Install dependencies
!pip install -q biosppy flirt ruamel.yaml hrv-analysis neurokit2 distinctipy joypy peakutils biopsykit lightning torchmetrics

if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)

### 1. Code Patching
Applying critical fixes to the codebase to ensure compatibility with the Kaggle environment.

In [ ]:
def apply_patch(file_path, old_str, new_str):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f: content = f.read()
        if old_str in content:
            content = content.replace(old_str, new_str)
            with open(file_path, 'w') as f: f.write(content)
            print(f"[PATCHED] {file_path}")

apply_patch('segment.py', 'newshape=', '')
apply_patch('src/timebase/data/preprocessing.py', 'newshape=', '')
apply_patch('src/timebase/data/reader.py', 'idx_dict_cases["train"] + idx_dict_controls["train"])', 'idx_dict_cases["train"] + idx_dict_controls["train"], dtype=np.int64)')

h5_p = 'src/timebase/utils/h5.py'
if os.path.exists(h5_p):
    with open(h5_p, 'r') as f: c = f.read()
    if 'compression="gzip"' not in c:
        c = c.replace('data=v,', 'data=v, compression="gzip", compression_opts=4,')
        with open(h5_p, 'w') as f: f.write(c)
        print("[PATCHED] h5.py compression")

# Patch COLLECTIONS_DICT in static.py
static_p = 'src/timebase/data/static.py'
if os.path.exists(static_p):
    with open(static_p, 'r') as f: c = f.read()
    if '"dati_preelaborati"' not in c:
        c = c.replace('"wesd": 12,', '"wesd": 12,\n    "dati_preelaborati": 0,')
        with open(static_p, 'w') as f: f.write(c)
        print("[PATCHED] static.py COLLECTIONS_DICT")

# Patch pre_train.py to avoid DoubleTensor vs FloatTensor error
pt_p = 'pre_train.py'
if os.path.exists(pt_p):
    with open(pt_p, 'r') as f: c = f.read()
    c = c.replace('v.to(device)', 'v.float().to(device)')
    with open(pt_p, 'w') as f: f.write(c)
    print("[PATCHED] pre_train.py float casting")

# Patch utils.py to show progress bar for verbose=1
u_p = 'src/timebase/data/utils.py'
if os.path.exists(u_p):
    with open(u_p, 'r') as f: c = f.read()
    c = c.replace('disable=args.verbose < 2', 'disable=args.verbose < 1')
    with open(u_p, 'w') as f: f.write(c)
    print("[PATCHED] utils.py progress")

### 2. Dataset Integration (Stable Remapping Logic)
Linking the pre-processed dataset and remapping the metadata registry.

In [ ]:
import os, glob, shutil, pickle

TARGET_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled'
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
os.makedirs(TARGET_DIR, exist_ok=True)

print("Searching for data in Input panel...")
meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files: 
    raise FileNotFoundError("ERROR: metadata.pkl not found!")
src_dir = os.path.dirname(meta_files[0])
print(f"Source: {src_dir}")

for item in os.listdir(src_dir):
    s, d = os.path.join(src_dir, item), os.path.join(TARGET_DIR, item)
    if item == 'metadata.pkl':
        shutil.copy2(s, d) 
    else:
        os.symlink(s, d)

meta_p = os.path.join(TARGET_DIR, 'metadata.pkl')
with open(meta_p, 'rb') as f: meta = pickle.load(f)

new_sessions = {}
found_count = 0
for root, dirs, files in os.walk(TARGET_DIR, followlinks=True):
    if 'channels.h5' in files:
        clean_name = os.path.basename(root)
        new_rel_id = os.path.relpath(root, TARGET_DIR).replace('\\\\', '/')
        for old_id in meta['sessions_info'].keys():
            if old_id.endswith(clean_name):
                new_sessions[new_rel_id] = meta['sessions_info'][old_id]
                found_count += 1
                break

if found_count > 0:
    meta['sessions_info'] = new_sessions
    with open(meta_p, 'wb') as f: pickle.dump(meta, f)
    print(f"LOADING COMPLETE! Mapped sessions: {found_count}")

### 3. Segmentation

In [ ]:
SEG_DIR = 'data/preprocessed/sl512_ss256_unlabelled'
print("Starting segmentation (sl512_ss256)...")
!PYTHONPATH=src python segment.py --data_dir {TARGET_DIR} --output_dir {SEG_DIR} --segment_length 512 --segmentation_mode 1 --step_size 256 --num_workers 2 --overwrite

### 4. Subject Split (70%)

In [ ]:
import pickle, os, numpy as np, re

print("Isolating Phase 1 Dataset (70% subjects)")
meta_path = os.path.join(SEG_DIR, "metadata.pkl")
with open(meta_path, "rb") as f: meta = pickle.load(f)

all_paths = meta["sessions_paths"]
sub_ids = []
for path in all_paths:
    p_norm = path.replace('\\', '/').replace('//', '/')
    parts = p_norm.split('/')
    final_sid = None
    for pt in parts:
        if 'sl512' in pt or 'ss256' in pt: continue
        match = re.search(r'[sS]ubject_?(\d+)|[sS](\d+)', pt)
        if match: 
            final_sid = f"{parts[0][:3].lower()}_{match.group(1) or match.group(2)}"
            break
    if not final_sid and len(parts) >= 2: final_sid = f"folder_{parts[-2]}"
    sub_ids.append(final_sid)

unique_subs = np.unique(sub_ids)
np.random.seed(42)
np.random.shuffle(unique_subs)
split_idx = int(len(unique_subs) * 0.70)
fase1_subs = set(unique_subs[:split_idx])

mask = np.array([s in fase1_subs for s in sub_ids], dtype=bool)
orig_size = len(all_paths)

# Filter all arrays consistently
for key in list(meta.keys()):
    if isinstance(meta[key], (list, np.ndarray)) and len(meta[key]) == orig_size:
        meta[key] = np.array(meta[key])[mask]
    elif isinstance(meta[key], dict):
        for sub_key in meta[key].keys():
            if isinstance(meta[key][sub_key], (list, np.ndarray)) and len(meta[key][sub_key]) == orig_size:
                meta[key][sub_key] = np.array(meta[key][sub_key])[mask]

with open(meta_path, "wb") as f: pickle.dump(meta, f)
print(f"Partitioned: {len(fase1_subs)} subjects, {np.sum(mask)} segments.")

### 5. SSL Pre-Training Execution

In [ ]:
print("Starting SSL Training...")
print("Note: Computing statistics for 17,000+ segments can take 10-15 minutes. Please wait.")
!PYTHONPATH=src python pre_train.py --dataset {SEG_DIR} --pretext_task masked_prediction --output_dir "output_pretrain" --epochs 50 --batch_size 32 --e4selflearning --verbose 1

### 6. Export Results
Archive and download the trained model and logs.

In [ ]:
import shutil
from IPython.display import FileLink

output_archive = "corponi_phase1_results"
shutil.make_archive(output_archive, 'gztar', "/kaggle/working/e4selflearning/output_pretrain")
print(f"Archive created: {output_archive}.tar.gz")
FileLink(f"{output_archive}.tar.gz")